<a href="https://colab.research.google.com/github/Azidian/Inteligencia_Artificial/blob/main/Componente_practico/Lecture_05/03_qlearning_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

**Respuesta:** Tiene 500 estados y 6 acciones.

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.


Porque no solo codifica la cuadrícula, cada estado único en el Taxi es una combinación de 4 variables: la posición del taxi (25 casillas) $\times$ la ubicación del pasajero (4 lugares posibles de origen + 1 estado "dentro del taxi") $\times$ el destino del pasajero (4 lugares posibles). Esto multiplica las posibilidades creando 500 escenarios distintos.   



## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?

- $s_t$: El estado actual en el que inició el taxi antes de moverse.   
- $a_t$: La acción (número del 0 al 5) que elegiste ejecutar.   
- $r_{t+1}$: La recompensa inmediata recibida por esa acción (generalmente -1 por paso, o -10 por recoger/dejar al pasajero ilegalmente).   
- $s_{t+1}$: El nuevo estado resultante en el que quedó el taxi tras la acción.


## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$

**Respuesta:**
$\vert{}S\vert{} \times \vert{}A\vert{} = 500 \times 6 = \mathbf{3000}$ valores en total.


## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    # Exploración: acción aleatoria
    if random.random() < epsilon:
        return env.action_space.sample()

    # Explotación: mejor acción con desempate aleatorio
    q_values = Q[state]
    max_q = np.max(q_values)
    best_actions = np.flatnonzero(q_values == max_q)
    return int(np.random.choice(best_actions))

## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    # En este entorno Gymnasium, next_state ya contempla todo.
    best_next_q = np.max(Q[next_state])
    td_target = reward + gamma * best_next_q
    td_error = td_target - Q[state, action]

    Q[state, action] += alpha * td_error

## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            # 1. escoger acción
            action = choose_action(Q, state, epsilon, env)

            # 2. ejecutar acción en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 3. actualizar Q
            update_q(Q, state, action, reward, next_state, alpha, gamma)

            # 4. actualizar estado y recompensa acumulada
            state = next_state
            total_reward += reward

            # 5. terminar si corresponde
            if done:
                break

        rewards.append(total_reward)

    return Q, rewards

## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?


> Sí, sube drásticamente desde un área muy negativa debido al límite de pasos y penalizaciones de -10 hacia valores cercanos a cero o ligeramente positivos.

> La curva empieza a estabilizarse alrededor de los 1500 a 2000 episodios, donde las fluctuaciones se reducen y la recompensa se mantiene cerca del máximo.

> Indica que se ha alcanzado una política óptima o muy cercana a la óptima, ya que el promedio de recompensas se estabiliza de manera plana en la parte superior del gráfico, maximizando la recompensa final en casi cada episodio.


## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?

> El comportamiento cambia bastante, porque al inicio, como no había explorado, se la pasaba dando vueltas al azar y se devolvía, alejándose de la meta o del lugar donde debía recoger al pasajero. En cambio, ya entrenado, va directo al grano.


- ¿El taxi sin entrenamiento logra completar la tarea?

> No, sin el entrenamiento no lograba completarla, porque en esa exploración perdía mucho tiempo devolviéndose. Como solo teníamos 50 pasos por episodio, se le acababan los intentos antes de poder llegar a la meta.

- ¿El agente entrenado evita acciones innecesarias?

> Así es, evita acciones innecesarias porque ya aprendió que dar pasos en falso le quita puntos. Por eso ya no pierde tiempo en movimientos que no lo acercan a su destino.


- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

> La evidencia visual más clara es que podemos ver al taxi ir directamente a recoger a la persona y llevarla hasta el edificio (la meta) sin quedarse atascado dando vueltas por el mapa.



## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?

La acción con mayor valor Q es la número 3 con un valor de 2.71224351.

2. ¿Qué significa que una acción tenga un valor Q mayor que otra?

Significa que es la mejor decisión que el agente puede tomar. Al observar la matriz, el agente elige esa acción con el valor mayor porque ya aprendió que las otras opciones no le darán un resultado tan bueno.

3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?

Porque el valor Q no solo mira los puntos que el agente gana en ese preciso momento, sino que también tiene en cuenta todas las recompensas futuras. Es decir, el agente piensa a largo plazo, a veces dar un paso no da puntos inmediatamente, pero lo pone en la posición correcta para completar la meta más adelante.

## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.


**Entrenamiento**

In [ ]:
Q_initial_exp = np.zeros((n_states, n_actions))

Q_trained_exp, rewards_exp = train_q_learning(
    env,
    Q_initial_exp.copy(),
    episodes=5000,
    alpha=0.8, # Modificado de 0.1 a 0.8
    gamma=0.95,
    epsilon=0.1
)

moving_average_exp = np.convolve(
    rewards_exp,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average, label="Original (alpha=0.1)")
plt.plot(moving_average_exp, label="Experimento (alpha=0.8)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Comparación de hiperparámetros ({window} episodios)")
plt.legend()
plt.show()

**Interpretación**

- Al aumentar el parámetro a 0.8, el aprendizaje inicial es más acelerado, subiendo la recompensa promedio más rápido al inicio.
- Aunque aprende más rápido, el modelo es más inestable y la curva puede presentar más fluctuaciones al estar sobreescribiendo fuertemente el conocimiento anterior con cada nueva experiencia.
- La recompensa final es muy similar, ya que la política alcanza un nivel óptimo o casi óptimo de cualquier forma.
- El agente es capaz de aprender a resolver la tarea en menos intentos, aunque los saltos en la estimación de los valores Q hacen que la gráfica no sea tan suave como en el caso original.

## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
